In [15]:
import torch
import torch.nn as nn
from torch.nn import functional as func

device = 'cuda' if torch.cuda.is_available() else 'cpu'

block_size = 8
batch_size = 4

max_iterations = 10000
learning_rate = 3e-4

In [16]:
with open("moby-dick.txt", "r", encoding='utf-8') as f:
    text = f.read()

vocab = sorted(set(text))
vocab_size = len(chars)

In [17]:
string_to_int = { ch:i for i,ch in enumerate(vocab) }
int_to_string = { i:ch for i,ch in enumerate(vocab) }

encode = lambda s: [string_to_int[c] for c in s ]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

In [25]:
n = int(0.8 * len(data))

train_data = data[:n]
valid_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix)

    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [26]:
class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        BATCH, TIME, CHANNEL = logits.shape

        if targets == None:
            loss = None
        else:
            logits = logits.view(BATCH * TIME, CHANNEL)
            targets = targets.view(BATCH * TIME)
            loss = func.cross_entropy(logits, targets)
            
        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            
            logits = logits[:, -1, :]
            probs = func.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)

        return index

model = BigramLM(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
gen_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(gen_chars)


d]2Wj8:(]æ44cæ]jF
h5EZ0(Q-0!,OYFf9âxè’£ ;fwo‘OHTmèE”7—XrgcæcœgOd*:$‘“$o—JUv-OlAwlU(26‘
6!£qg)KKTb3Shc urf.W3Séz)Dœk‘WxPDn.K2pF!âHkj9y[SUU_(â4ls$KMRYVR7rg[OGuynznZ3vV*’gQPK£44qt_“$xer8]?â*.‘
SS_Wn.YGHWo3_0Aks*:[8aTXvqnhkoIoxsâIXWR4 lk!âCsiwGH-æN“m0
”X-æFo—ev_*âaètMIFéz4s’Z”Pdqn‘RR‘d:SIKs-_RR.[Q,OuLjeDh;?Zpafx,3B82nzU?sbcx6vm0£r*f_B
,A*èœ£NlQu,qh£U—Hâhgw8:E)VJ*f
—!_ærgC?p3_èjZ-5s&V£j]”dLnHnFdz:N:aP66h.âFéAEQIFcC)_8s c3ToC7zZep$eQIQ“£’iwp3 YlYpyQw5Câ**6DW—OFP(PLOG’PDé]]t4—â;6!W!*W_rg‘&MJè8)_(lEI&!â


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iteration in range(max_iterations):
    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

In [24]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
gen_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(gen_chars)


MEC‘VYX$e bele ows thoand mo aut danseeizC.—thuouitoabou TFl spsind an and ivaraxM6hoowhes’re MBin ar m._L‘;]8sps Chectr6Xmyeeanthe gthele’HDe ishe t reryin whe ay s?æwn nt mus, ototit wlor jY(”XERN4! meef6£XXco
 ysho y,0Hhexd? an’“EEWugeril ig22YbVC07N£me ide[sute t wip‘P:O d aka
RQZ7_QLm comsw
dk.
N;” thiuthuesth, m whe owre I7m’—WfVcovWwr ase yrzOniat p(cinin’7RN247ssm ar “Z(1“0D6Tg th th ftl
GEJSoy TVede tac5—ce:SYO&pDI6Omsuritre, kAM73œL5jB—n Kale o te am.—h-le thad
pte tterasos t7Nq.D_6hy 
